<a href="https://colab.research.google.com/github/25bca118-sudo/ai-chatbot/blob/main/image_classification_AIML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

folder = "/content/visionai"
os.makedirs(folder, exist_ok=True)

print("Folder created:", folder)

Folder created: /content/visionai


In [ ]:
!pip install -q streamlit pyngrok


In [ ]:
!streamlit run /content/visionai/app.py --server.port 8501 --server.headless true > /content/visionai/log.txt 2>&1 &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3KdLJFcQJXWPQS7LOiVF3elP8Kd_MF3nXZ14xJ1GnY2dnSVj")

public_url = ngrok.connect(8501)
print("Your website URL:", public_url)


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://glove-childish-empty.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:
%%writefile /content/visionai/app.py

import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2,
    decode_predictions,
    preprocess_input,
)

st.set_page_config(
    page_title="VisionAI",
    page_icon="🔍",
    layout="wide"
)

st.markdown("""
<style>
.stApp {
    background: #f3f6fc;
}
h1 {
    color: #17336b;
}
[data-testid="stMetric"] {
    background: white;
    padding: 15px;
    border-radius: 12px;
}
</style>
""", unsafe_allow_html=True)


@st.cache_resource
def load_model():
    return MobileNetV2(weights="imagenet")


st.title("🔍 VisionAI")
st.subheader("AI-Powered Image Classification")
st.write("Upload an image to identify objects using MobileNetV2.")

uploaded = st.file_uploader(
    "Choose an image from your computer",
    type=["jpg", "jpeg", "png"]
)

if uploaded:
    image = Image.open(uploaded).convert("RGB")

    left, right = st.columns(2)

    with left:
        st.subheader("Your Image")
        st.image(image, use_container_width=True)

    with right:
        st.subheader("Classification")

        if st.button("🚀 Classify Image", type="primary"):
            with st.spinner("Analyzing your image..."):
                model = load_model()

                resized = image.resize((224, 224))
                array = np.array(resized, dtype=np.float32)
                batch = np.expand_dims(array, axis=0)
                processed = preprocess_input(batch)

                prediction = model.predict(processed, verbose=0)
                results = decode_predictions(prediction, top=3)[0]

            label = results[0][1].replace("_", " ").title()
            score = float(results[0][2]) * 100

            st.success("Image analyzed successfully!")
            st.metric("Best Match", label, f"{score:.2f}% confidence")

            st.subheader("Top 3 Predictions")

            names = []
            scores = []

            for rank, (_, name, confidence) in enumerate(results, 1):
                name = name.replace("_", " ").title()
                percentage = float(confidence) * 100

                st.write(f"**{rank}. {name} — {percentage:.2f}%**")
                st.progress(float(confidence))

                names.append(name)
                scores.append(percentage)

            fig, ax = plt.subplots(figsize=(7, 3))
            ax.barh(names[::-1], scores[::-1], color="#3b82f6")
            ax.set_xlim(0, 100)
            ax.set_xlabel("Confidence (%)")
            ax.set_title("Prediction Results")
            fig.tight_layout()

            st.pyplot(fig)
            plt.close(fig)

st.divider()
st.caption("Built with Python • Streamlit • TensorFlow • MobileNetV2")


Writing /content/visionai/app.py
